# `semantic.v_growth_of_100` — view

Thin view over Gold. No logic beyond shaping.

A view is its own definition, so there is no load step and no etl task to pair with this one.

In [0]:
-- THE CHART. What £100 would have become, month by month, for every ticker.
-- A running product in log space again: adding logarithms is multiplying, and SQL has no
-- running product. This is the one view the dashboard plots as a line.
CREATE OR REPLACE VIEW `index-vs-trust-pipeline`.semantic.v_growth_of_100
COMMENT 'Value of 100 invested at the start of the study window, monthly, per ticker'
AS
SELECT f.ticker,
       CASE WHEN d.entity_type = 'Index' THEN CONCAT(d.trust_name, ' (the index)')
            ELSE d.trust_name END                      AS name,
       d.entity_type,
       d.management_group,
       d.status,
       f.month_key,
       dd.month_start,
       dd.month_label,
       dd.market_event,
       ROUND(100 * f.total_return, 2)                  AS monthly_return_pct,
       ROUND(100 * EXP(SUM(LN(1 + COALESCE(f.total_return, 0))) OVER (
                       PARTITION BY f.ticker ORDER BY f.month_key
                       ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)), 2)
                                                       AS value_of_100
FROM `index-vs-trust-pipeline`.gold.fact_monthly_performance f
JOIN `index-vs-trust-pipeline`.gold.dim_ticker d ON d.ticker_key = f.ticker_key
JOIN `index-vs-trust-pipeline`.gold.dim_date   dd ON dd.month_key = f.month_key
WHERE f.return_basis = 'total';

## Verification

Expected: the view resolves and returns rows. Counts are in 
`
specs/04_semantic/dashboard.md
`
.

In [0]:
SELECT COUNT(*) AS rows
FROM `index-vs-trust-pipeline`.semantic.v_growth_of_100;